In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 21:57:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [13]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())


Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [5]:
# A. Feature Engineering
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Mengubah kolom dengan StringIndexer
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

model_menu = indexer_menu.fit(df)
df_idx = model_menu.transform(df)

model_cabang = indexer_cabang.fit(df_idx)
df_idx = model_cabang.transform(df_idx)

model_waktu = indexer_waktu.fit(df_idx)
df_idx = model_waktu.transform(df_idx)

# Menggabungkan fitur numerik dan hasil index ke dalam satu kolom vector
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

df_final = assembler.transform(df_idx)
df_final.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



In [6]:
# B. Train/Test Split & Pelatihan Model
from pyspark.ml.regression import LinearRegression

# Split data 75% training, 25% testing dengan seed 101
train, test = df_final.randomSplit([0.75, 0.25], seed=101)

# Inisialisasi dan melatih model Linear Regression utama
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr.fit(train)

print("Koefisien fitur:", model_utama.coefficients)
print("Intercept:", model_utama.intercept)

Koefisien fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [8]:
# C. Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator

# Membuat prediksi menggunakan data testing
predictions = model_utama.transform(test)

# Evaluasi RMSE
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# Evaluasi R²
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

print("\nNilai R² yang sangat tinggi (mendekati 1) menunjukkan bahwa model regresi ini sudah sangat baik dan akurat untuk dipakai memprediksi omzet. Hal ini wajar karena fitur utama (qty dikali harga_satuan) memiliki korelasi matematis yang sangat kuat dengan target total_penjualan.")

RMSE : Rp 10,901
R²   : 0.9107

Nilai R² yang sangat tinggi (mendekati 1) menunjukkan bahwa model regresi ini sudah sangat baik dan akurat untuk dipakai memprediksi omzet. Hal ini wajar karena fitur utama (qty dikali harga_satuan) memiliki korelasi matematis yang sangat kuat dengan target total_penjualan.


In [10]:
# D. Eksperimen Perbandingan Fitur
# Membuat assembler baru yang hanya menggunakan qty dan harga_satuan
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"], 
    outputCol="features_sederhana"
)

df_sederhana = assembler_sederhana.transform(df_idx)

# Melakukan split ulang pada dataframe sederhana
train_sed, test_sed = df_sederhana.randomSplit([0.75, 0.25], seed=101)

# Melatih model kedua
lr_sed = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_sed = lr_sed.fit(train_sed)

# Evaluasi model kedua
pred_sed = model_sed.transform(test_sed)
r2_sed = evaluator_r2.evaluate(pred_sed)

print(f"R² Model Utama (5 fitur)  : {r2:.4f}")
print(f"R² Model Sederhana (2 fitur): {r2_sed:.4f}")

print("\nModel kedua yang hanya menggunakan qty dan harga_satuan memiliki nilai R² yang nyaris sama dengan model utama. Ini membuktikan bahwa menambahkan informasi nama_menu, cabang, dan waktu_pesan tidak memberikan peningkatan akurasi prediksi secara berarti, karena nilai dasar omzet secara dominan hanya dibentuk oleh kuantitas dikalikan harganya.")

R² Model Utama (5 fitur)  : 0.9107
R² Model Sederhana (2 fitur): 0.9108

Model kedua yang hanya menggunakan qty dan harga_satuan memiliki nilai R² yang nyaris sama dengan model utama. Ini membuktikan bahwa menambahkan informasi nama_menu, cabang, dan waktu_pesan tidak memberikan peningkatan akurasi prediksi secara berarti, karena nilai dasar omzet secara dominan hanya dibentuk oleh kuantitas dikalikan harganya.


In [11]:
# E. Prediksi Skenario Baru
import pandas as pd

# 1. Membuat DataFrame data dummy
skenario_data = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Roti Bakar", "Es Teh"],
    "cabang": ["Magelang", "Semarang", "Yogyakarta"],
    "waktu_pesan": ["Pagi", "Malam", "Siang"],
    "qty": [3, 2, 5],
    "harga_satuan": [20000, 22000, 15000]
})

df_skenario = spark.createDataFrame(skenario_data)

# 2. Menggunakan model StringIndexer yang SUDAH DI-FIT dari Bagian A
df_sken_idx = model_menu.transform(df_skenario)
df_sken_idx = model_cabang.transform(df_sken_idx)
df_sken_idx = model_waktu.transform(df_sken_idx)

# 3. Transformasi fitur menggunakan VectorAssembler dari Bagian A
df_sken_final = assembler.transform(df_sken_idx)

# 4. Melakukan prediksi menggunakan model utama
prediksi_skenario = model_utama.transform(df_sken_final)

# 5. Menampilkan hasil
prediksi_skenario.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show()

+----------+----------+-----------+---+------------+------------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|        prediction|
+----------+----------+-----------+---+------------+------------------+
| Kopi Susu|  Magelang|       Pagi|  3|       20000| 59572.01525544525|
|Roti Bakar|  Semarang|      Malam|  2|       22000|44099.758797337854|
|    Es Teh|Yogyakarta|      Siang|  5|       15000|  89262.8710599834|
+----------+----------+-----------+---+------------+------------------+



In [14]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
